# Online Recruitment Fraud Detection - Machine Learning Baselines
This notebook implements the traditional Machine Learning baselines (Logistic Regression and Random Forest) using TF-IDF features on the EMSCAD dataset.

### Setup Instructions:
1. Upload the dataset `fake_job_postings.csv` to the left files panel in Google Colab (drag-and-drop), or place it in the same directory as this notebook.
2. Run all cells sequentially.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

csv_path = "fake_job_postings.csv"

if not os.path.exists(csv_path):
    try:
        from google.colab import files
        print("Dataset 'fake_job_postings.csv' not found. Please upload it:")
        uploaded = files.upload()
    except ImportError:
        print(f"Local file '{csv_path}' not found. Please place it in the same directory.")
else:
    print(f"Dataset found at '{csv_path}'. Skipping upload prompt.")

## Load and Preprocess Dataset

In [ ]:
print("Loading dataset...")
df = pd.read_csv(csv_path)
print(f"Dataset shape: {df.shape}")

# Fill missing values with empty strings
df.fillna("", inplace=True)

# Concatenate key text columns to form a single context-aware field
df['combined_text'] = df['title'] + " " + df['company_profile'] + " " + df['description'] + " " + df['requirements'] + " " + df['benefits']

# Print class distribution
print("Class distribution (0 = Genuine, 1 = Fraudulent):")
print(df['fraudulent'].value_counts(normalize=True))

## Train/Test Split (Stratified)

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df['fraudulent'], random_state=42
)
print(f"Train set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

## TF-IDF Feature Extraction

In [ ]:
print("Vectorizing text using TF-IDF...")
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train = vectorizer.fit_transform(train_df['combined_text'])
X_test = vectorizer.transform(test_df['combined_text'])
y_train = train_df['fraudulent'].values
y_test = test_df['fraudulent'].values
print("TF-IDF shape:", X_train.shape)

## 1. Logistic Regression

In [ ]:
print("Training Logistic Regression...")
# Use balanced class weights to help with imbalance
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000)
lr_model.fit(X_train, y_train)

lr_preds = lr_model.predict(X_test)
lr_probs = lr_model.predict_proba(X_test)[:, 1]

print("Logistic Regression Report:")
print(classification_report(y_test, lr_preds))

## 2. Random Forest

In [ ]:
print("Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest Report:")
print(classification_report(y_test, rf_preds))

## Evaluate and Plot Results

In [ ]:
# Plot Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(confusion_matrix(y_test, lr_preds), annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Logistic Regression')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(confusion_matrix(y_test, rf_preds), annot=True, fmt='d', cmap='Oranges', ax=axes[1])
axes[1].set_title('Random Forest')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

# Plot ROC Curves
plt.figure(figsize=(8, 6))
for name, probs in [("Logistic Regression", lr_probs), ("Random Forest", rf_probs)]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.4f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Machine Learning Baselines')
plt.legend(loc='lower right')
plt.show()